In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet('/home/zengjunyao/model_link/af/sample.parquet')

In [3]:
df
print(df.shape)

(69534, 99)


In [4]:
#筛选新客--0、老客--1、半新--2样本
df_firstloan = df[df['new_old_user_status'].isin([1,2]) ]
#筛选状态为正常还款的样本（剔除展期订单settlement_type==2（经验））  settlement_type：结清方式（1-正常还款 2-展期 3-销账）
df_firstloan = df_firstloan[df_firstloan['settlement_type'] != 2]
#按日期筛选订单
df_firstloan = df_firstloan[df_firstloan['loan_date'] >= '2025-01-01']
df_firstloan = df_firstloan[df_firstloan['loan_date'] <= '2025-04-04']

In [5]:
#新客建模需对user_id去重
#df_firstloan.drop_duplicates(subset = ['user_id'],inplace=True)

In [6]:
#因为建模此次为纯投样本建模，筛选source=2/3的样本。
#df_firstloan = df_firstloan[(df_firstloan['source'] == '2') | (df_firstloan['source'] == '3')]

In [7]:
#定义模型的标签,def_pd0为以逾期0天作为标签，agr_pd0为筛选此标签下的到期订单。
df_firstloan['label'] = df_firstloan['def_pd0']
df_firstloan = df_firstloan[df_firstloan['agr_pd0'] == 1]

In [8]:
#索引重置为[0,1,2,...]
df_firstloan.index=range(df_firstloan.shape[0])

In [25]:
#切分样本为训练集与oot,一般训练集与oot的比例为0.75:0.25。
df_firstloan['sample_type'] = 'train'   #创建一个新的列sample_type,暂时将值全标记为'train'

In [26]:
df_firstloan.shape[0]*0.75

38877.0

In [27]:
df_firstloan.loc[:38877,'loan_date']  #找划分时间点02-27/02-28

0        2025-01-01
1        2025-01-01
2        2025-01-01
3        2025-01-01
4        2025-01-01
            ...    
38873    2025-03-01
38874    2025-03-01
38875    2025-03-01
38876    2025-03-01
38877    2025-03-01
Name: loan_date, Length: 38878, dtype: object

In [28]:
#df_firstloan['sample_type'][df_firstloan['loan_date'] >= '2025-02-28'] = 'oot' #从sample_type列中筛选出贷款日期>=2025-02-28的样本标记为'oot'
df_firstloan.loc[df_firstloan['loan_date']>='2025-03-02','sample_type'] = 'oot'

In [29]:
df_firstloan['sample_type'].value_counts()

sample_type
train    39370
oot      12466
Name: count, dtype: int64

In [30]:
df_firstloan.to_pickle('df_sample_v11.pkl')

In [31]:
a = pd.read_pickle('df_sample_v11.pkl')

In [32]:
a

,id,user_id,acq_channel,app_order_id,contract_no,repayment_date,installment_num,installment_amount,principal,interest,...,profit,original,is_extension,repaid_ltv,payout_ltv,profit_ltv,def_ltv,file_name,label,sample_type
0,244467,1317640458664894464,VIJG,1323701931170291712,1323702035390357504,2025-01-07,1,2200.0,2200.0,0.0,...,880.0,1,0,37159.0,24063.0,13096.0,0,df_1_2025-01-01.parquet,0,train
1,244468,1261698652018532352,APFL,1323702219562246144,1323702422545588224,2025-01-07,1,3200.0,3200.0,0.0,...,1280.0,1,0,53200.0,44648.0,8552.0,0,df_1_2025-01-01.parquet,0,train
2,244469,1258062236822757376,HLOA,1323703298987352064,1323703479929626624,2025-01-07,1,3050.0,3050.0,0.0,...,1220.0,1,0,81050.0,63432.0,17618.0,0,df_1_2025-01-01.parquet,0,train
3,244472,1258062236822757376,HLOA,1323706215282728960,1323706382891311104,2025-01-07,1,5500.0,5500.0,0.0,...,2199.0,1,0,78000.0,61602.0,16398.0,0,df_1_2025-01-01.parquet,0,train
4,244474,1318174204879134720,VIJG,1323669295601246208,1323677253802352640,2025-01-07,1,2800.0,2800.0,0.0,...,1120.0,1,0,8200.0,12632.0,-4432.0,1,df_2_2025-01-01.parquet,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51831,341770,1297505747472965632,VIJG,1357734941040500736,1357735203289358336,2025-04-10,1,3700.0,3700.0,0.0,...,-2221.0,1,0,0.0,2221.0,-2221.0,1,df_1_2025-04-04.parquet,1,oot
51832,341772,1328698032113803264,VIJG,1357741567382315008,1357741815718666240,2025-04-10,1,1600.0,1600.0,0.0,...,-960.0,1,0,0.0,1920.0,-1920.0,1,df_2_2025-04-04.parquet,1,oot
51833,341773,1328698032113803264,VIJG,1357741567382315009,1357741848971108352,2025-04-10,1,1600.0,1600.0,0.0,...,-960.0,1,0,0.0,960.0,-960.0,1,df_2_2025-04-04.parquet,1,oot
51834,341775,1291054282550239232,APFL,1357747537718206464,1357747835765448704,2025-04-10,1,3200.0,3200.0,0.0,...,1280.0,1,0,6800.0,13562.0,-6762.0,1,df_1_2025-04-04.parquet,0,oot
